# 04 Global DeepAR Probabilistic Forecasting

This notebook trains the Phase 2 standalone neural forecast. It does not change any Phase 1 file.

**DeepAR idea:** one small LSTM learns from all item-store series. At each day it receives the previous demand, known features, and static product/store embeddings. It predicts a Negative Binomial distribution instead of one number.

The Negative Binomial is appropriate here because sales are non-negative counts and their variance is much larger than their mean. We extract q10, q50 and q90 from recursive sample paths.

## 1. Locate the project

The notebook may be launched from the repository root or from `Notebooks/`. This cell finds the root and makes the tested Phase 2 modules importable.

In [3]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
print("Project root located; Phase 2 modules are importable.")

Project root located; Phase 2 modules are importable.


## 2. Load configuration and prepare data

The input table has one row per series and date. The target is daily `sales`. Calendar and price features are joined without changing Phase 1. A price-present indicator is only an **availability proxy**; it is not verified inventory or a verified stockout.

In [4]:
from src.phase2.experiments import prepare_experiment_data

data, windows, boundaries, category_maps, feature_set, config = prepare_experiment_data(
    PROJECT_ROOT,
    experiment_name="deepar",
)

print("Rows:", len(data))
print("Series:", data["id"].nunique())
print("Dynamic features:", feature_set["dynamic"])
print("Static features:", feature_set["static"])
print(windows)
print(boundaries)

Rows: 191300
Series: 100
Dynamic features: ['day_of_week_scaled', 'month_scaled', 'is_weekend', 'event', 'snap', 'log_price', 'price_change', 'relative_price', 'is_available_proxy', 'is_price_missing', 'is_store_closed', 'observation_mask']
Static features: ['item_id', 'store_id', 'cat_id', 'dept_id', 'state_id']
   window     origin forecast_start forecast_end
0       1 2016-03-27     2016-03-28   2016-04-03
1       2 2016-04-03     2016-04-04   2016-04-10
2       3 2016-04-10     2016-04-11   2016-04-17
3       4 2016-04-17     2016-04-18   2016-04-24
{'training_end': Timestamp('2016-03-13 00:00:00'), 'validation_start': Timestamp('2016-03-14 00:00:00'), 'validation_end': Timestamp('2016-03-27 00:00:00'), 'test_start': Timestamp('2016-03-28 00:00:00'), 'test_end': Timestamp('2016-04-24 00:00:00')}


## 3. Inspect availability assumptions

`observation_mask = 1` means a selling price exists and the store is not marked closed for Christmas. Zero sales with a valid price remain genuine observed zeros. We never silently turn ambiguous zero sales into missing targets.

In [5]:
availability_summary = data[[
    "is_price_missing",
    "is_available_proxy",
    "is_zero_sales",
    "is_leading_zero",
    "observation_mask",
]].mean()

display(availability_summary.to_frame("fraction of rows"))

,fraction of rows
is_price_missing,0.173298
is_available_proxy,0.824438
is_zero_sales,0.646864
is_leading_zero,0.173298
observation_mask,0.824438


## 4 Construct one sequence

A batch sent to the LSTM has shape `(batch_size, sequence_length, number_of_features)`.

- `batch_size`: independent training examples processed together.
- `sequence_length`: 56 context days plus 7 forecast days = 63.
- `number_of_features`: the dynamic inputs for each day.

During training, actual previous sales are inputs. This is teacher forcing. The target at day `t` never enters the input for day `t`; the input contains sales from day `t-1`. The loss mask selects only the final seven target days.

In [6]:
from src.phase2.deepar_model import make_scale_map, make_training_arrays

scale_map = make_scale_map(data, boundaries["training_end"])
training_arrays = make_training_arrays(
    data=data,
    training_end=boundaries["training_end"],
    context_length=config["context_length"],
    horizon=config["forecast_horizon"],
    stride=config["training_stride"],
    dynamic_features=feature_set["dynamic"],
    static_columns=feature_set["static"],
    scale_map=scale_map,
)

print("Previous-sales tensor:", training_arrays["previous_sales"].shape)
print("Dynamic-feature tensor:", training_arrays["dynamic"].shape)
print("Static-category tensor:", training_arrays["static"].shape)
print("Target tensor:", training_arrays["target"].shape)
print("Loss-mask tensor:", training_arrays["loss_mask"].shape)
print("First example forecast mask:", training_arrays["loss_mask"][0, -10:])

Previous-sales tensor: (13100, 63)
Dynamic-feature tensor: (13100, 63, 12)
Static-category tensor: (13100, 5)
Target tensor: (13100, 63)
Loss-mask tensor: (13100, 63)
First example forecast mask: [0. 0. 0. 1. 1. 1. 1. 1. 1. 1.]


## 5 Prepare a batch

The dataset converts NumPy arrays to PyTorch tensors. A data loader groups examples into a batch. Its dynamic tensor has shape `(batch size, 63 days, dynamic features)`. Static identifiers have no time dimension because one product and store identity applies to the whole sequence.

In [7]:
from torch.utils.data import DataLoader
from src.phase2.deepar_model import SequenceDataset

demo_loader = DataLoader(SequenceDataset(training_arrays), batch_size=4, shuffle=False)
demo_batch = next(iter(demo_loader))
for name, tensor in demo_batch.items():
    print(f"{name:>14}: {tuple(tensor.shape)}")

previous_sales: (4, 63)
       dynamic: (4, 63, 12)
        static: (4, 5)
        target: (4, 63)
     loss_mask: (4, 63)
         scale: (4,)


## 6 Pass the sequence through the LSTM

The model first scales and transforms previous demand with `log1p`. It concatenates that value with dynamic features and repeated static embeddings. The LSTM returns one hidden representation per day. Two linear heads turn each hidden vector into positive distribution parameters by applying `softplus`.

The first parameter is the Negative Binomial mean. The second is dispersion. In this implementation, `variance = mean + dispersion × mean²`. Positive dispersion lets the variance exceed the mean, which is common for intermittent retail counts.

In [8]:
import torch
from src.phase2.deepar_model import SimpleDeepAR

cardinalities = {name: len(category_maps[name]) + 1 for name in feature_set["static"]}
demo_model = SimpleDeepAR(
    number_of_dynamic_features=len(feature_set["dynamic"]),
    static_cardinalities=cardinalities,
    hidden_size=config["hidden_size"],
    number_of_layers=config["number_of_lstm_layers"],
    embedding_size=config["embedding_size"],
)
mean, dispersion, hidden = demo_model(
    demo_batch["previous_sales"], demo_batch["dynamic"],
    demo_batch["static"], demo_batch["scale"],
)
print("Mean shape:", tuple(mean.shape))
print("Dispersion shape:", tuple(dispersion.shape))
print("LSTM hidden-state shape:", tuple(hidden[0].shape))
print("All parameters positive:", bool((mean > 0).all() and (dispersion > 0).all()))

Mean shape: (4, 63)
Dispersion shape: (4, 63)
LSTM hidden-state shape: (1, 4, 24)
All parameters positive: True


## 7 Calculate the Negative Binomial loss and update weights

The Negative Binomial assigns a probability to each observed count. Training minimizes negative log likelihood, so the model is penalized when it assigns low probability to the actual demand. Multiplication by the loss mask excludes the context days. `backward()` calculates gradients, gradient clipping limits unstable updates, and `optimizer.step()` changes the weights.

In [9]:
from src.phase2.deepar_model import masked_negative_binomial_loss

optimizer = torch.optim.Adam(demo_model.parameters(), lr=config["learning_rate"])
optimizer.zero_grad()
loss = masked_negative_binomial_loss(
    mean, dispersion, demo_batch["target"], demo_batch["loss_mask"]
)
loss.backward()
gradient_norm = torch.nn.utils.clip_grad_norm_(
    demo_model.parameters(), config["gradient_clip_norm"]
)
optimizer.step()
print("One teaching-step loss:", float(loss.detach()))
print("Gradient norm before clipping:", float(gradient_norm))

One teaching-step loss: 0.9307676553726196
Gradient norm before clipping: 0.5188136696815491


## 8 Train the full model

The LSTM hidden state is a learned numerical memory of recent demand and covariates. Two output heads produce the mean and dispersion of a Negative Binomial distribution. At prediction time, sampled demand is fed back into the next step, so the seven-day forecast is genuinely autoregressive.

The model trains only through `training_end`, selects its checkpoint using the following 14 validation days, and evaluates on the untouched final 28 days.

In [10]:
from src.phase2.experiments import run_training_experiment

result = run_training_experiment(
    project_root=PROJECT_ROOT,
    experiment_name="deepar",
    experiment_id="deepar_base_v2",
    model_name="DeepAR + price + availability",
)

display(result["history"])

,epoch,training_loss,validation_loss
0,1,1.316306,1.169442
1,2,1.216322,1.152367
2,3,1.200829,1.144393
3,4,1.192281,1.146047
4,5,1.190280,1.142286
5,6,1.184531,1.140708


## 9 Generate recursive sample paths and quantiles

At the forecast origin, the LSTM reads the observed context once. For forecast step one, it samples a count from the predicted Negative Binomial distribution. That sampled count becomes the previous-demand input for step two. Repeating this process creates a sample path without using future actual sales. One hundred paths form an empirical forecast distribution. The 10th, 50th, and 90th percentiles become q10, q50, and q90.

In [11]:
first_series_id = result["predictions"]["id"].iloc[0]
one_series = result["predictions"][result["predictions"]["id"].eq(first_series_id)]
display(one_series[["forecast_origin", "forecast_date", "actual", "q10", "q50", "q90"]].head(7))
print("Quantiles ordered:", bool(
    (one_series["q10"] <= one_series["q50"]).all()
    and (one_series["q50"] <= one_series["q90"]).all()
))

,forecast_origin,forecast_date,actual,q10,q50,q90
0,2016-03-27,2016-03-28,0.0,0.0,0.0,0.0
1,2016-03-27,2016-03-29,0.0,0.0,0.0,1.0
2,2016-03-27,2016-03-30,0.0,0.0,0.0,1.0
3,2016-03-27,2016-03-31,0.0,0.0,0.0,1.0
4,2016-03-27,2016-04-01,0.0,0.0,0.0,1.0
5,2016-03-27,2016-04-02,0.0,0.0,0.0,1.0
6,2016-03-27,2016-04-03,0.0,0.0,0.0,1.0


Quantiles ordered: True


## 10 Evaluate point accuracy and uncertainty

q50 is median predicted demand. q10 and q90 bound an 80% central interval. Over many representative forecasts, an 80% interval should cover roughly 80% of actual values, but useful intervals must also remain narrow.

In [12]:
metrics = result["metrics"]
display(result["predictions"].head(10))
display({name: value for name, value in metrics.items() if name != "device"})

,experiment_id,Model,id,item_id,store_id,cat_id,forecast_origin,forecast_date,horizon_step,actual,q10,q50,q90
0,deepar_base_v2,DeepAR + price + availability,FOODS_1_026_TX_3_validation,FOODS_1_026,TX_3,FOODS,2016-03-27,2016-03-28,1,0.0,0.0,0.0,0.0
1,deepar_base_v2,DeepAR + price + availability,FOODS_1_026_TX_3_validation,FOODS_1_026,TX_3,FOODS,2016-03-27,2016-03-29,2,0.0,0.0,0.0,1.0
2,deepar_base_v2,DeepAR + price + availability,FOODS_1_026_TX_3_validation,FOODS_1_026,TX_3,FOODS,2016-03-27,2016-03-30,3,0.0,0.0,0.0,1.0
3,deepar_base_v2,DeepAR + price + availability,FOODS_1_026_TX_3_validation,FOODS_1_026,TX_3,FOODS,2016-03-27,2016-03-31,4,0.0,0.0,0.0,1.0
4,deepar_base_v2,DeepAR + price + availability,FOODS_1_026_TX_3_validation,FOODS_1_026,TX_3,FOODS,2016-03-27,2016-04-01,5,0.0,0.0,0.0,1.0
5,deepar_base_v2,DeepAR + price + availability,FOODS_1_026_TX_3_validation,FOODS_1_026,TX_3,FOODS,2016-03-27,2016-04-02,6,0.0,0.0,0.0,1.0
6,deepar_base_v2,DeepAR + price + availability,FOODS_1_026_TX_3_validation,FOODS_1_026,TX_3,FOODS,2016-03-27,2016-04-03,7,0.0,0.0,0.0,1.0
7,deepar_base_v2,DeepAR + price + availability,FOODS_1_026_TX_3_validation,FOODS_1_026,TX_3,FOODS,2016-04-03,2016-04-04,1,0.0,0.0,0.0,2.0
8,deepar_base_v2,DeepAR + price + availability,FOODS_1_026_TX_3_validation,FOODS_1_026,TX_3,FOODS,2016-04-03,2016-04-05,2,0.0,0.0,0.0,1.0
9,deepar_base_v2,DeepAR + price + availability,FOODS_1_026_TX_3_validation,FOODS_1_026,TX_3,FOODS,2016-04-03,2016-04-06,3,0.0,0.0,0.0,1.0


{'MAE': 0.9221428571428572,
 'RMSE': 2.1194254612310113,
 'WAPE_percent': 60.45422617653945,
 'pinball_q10': 0.12771785714285716,
 'pinball_q50': 0.4610714285714286,
 'pinball_q90': 0.3451392857142857,
 'mean_pinball_loss': 0.3113095238095238,
 'interval_coverage_80_percent': 93.89285714285714,
 'empirical_q10_below_percent': 0.75,
 'empirical_q90_at_or_below_percent': 94.64285714285714,
 'empirical_q90_above_percent': 5.357142857142857,
 'mean_interval_width': 3.3714285714285728,
 'forecast_points': 2800,
 'experiment_id': 'deepar_base_v2',
 'Model': 'DeepAR + price + availability',
 'training_seconds': 6.9359045829987735,
 'training_examples': 13100,
 'epochs_completed': 6}

## 11 Saved outputs

The checkpoint, configuration, training history, predictions, aggregate metrics and calibration summaries are saved under `models/phase2/` and `results/phase2/`. The prediction table contains one row per series, origin and forecast day.

In [13]:
output_files = sorted((PROJECT_ROOT / "results" / "phase2").glob("deepar_base_v2*"))
for path in output_files:
    print(path.relative_to(PROJECT_ROOT))

results/phase2/deepar_base_v2_calibration.csv
results/phase2/deepar_base_v2_metrics.csv
results/phase2/deepar_base_v2_predictions.csv.gz
results/phase2/deepar_base_v2_training_history.csv
